# Agent AI Bootcamp

Code from the section notes goes into cells below. Keys are read from `.env` in the project root.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# Groq (OpenAI-compatible). To use OpenAI instead, replace with: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

Quick API check. Run this after putting your real Groq key in `.env`; a 401 error means the key is wrong or still the placeholder.

In [2]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Reply with the single word: ok"}],
)
print(response.choices[0].message.content)

ok


## 03 RAG: Retrieval-Augmented Generation

Course notes use OpenAI `responses.create` with `gpt-5.4-mini`. We use Groq, so the same idea is written with `chat.completions` and `openai/gpt-oss-120b`.

In [3]:
MODEL = "openai/gpt-oss-120b"

def llm(prompt):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [4]:
llm("Hey, what's up?")

'Hey! I’m doing great—thanks for asking. How about you? Anything fun or interesting on your mind today?'

### Plain LLM lacks our data
A course-specific question gets a generic answer: the model knows nothing about Zoomcamp enrollment.

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! 🎉 Most of our courses are open for enrollment at any time, and we’d love to have you join.

**Here’s what you can do to get started:**

1. **Pick the course** – Let us know which specific course you’re interested in (e.g., *Introduction to Python, Digital Marketing Basics, Data Science Fundamentals*, etc.).
2. **Check the schedule** – Some courses have live sessions on specific days/times, while others are self‑paced. I can send you the upcoming calendar or the link to the recorded modules.
3. **Enroll** – Once you’ve decided, you can register directly on our website or I can give you a quick enrollment link.
4. **Get the materials** – After you’ve signed up, you’ll receive access to the syllabus, any required reading, and a welcome packet with all the details you need to hit the ground running.

If you’d like, I can:

- Send you the **full course catalog**.
- Provide the **enrollment link** for the course you have in mind.
- Answer any questions about prerequisites, pricin

### Adding context manually
Paste FAQ entries into the prompt. This is RAG done by hand.

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just make sure to submit your project before the submission deadline.


### RAG = search + prompt + LLM

```
User question -> search(knowledge base) -> top documents -> build_prompt(question, docs) -> llm(prompt) -> answer
```

The LLM only sees the documents we hand it, so answers are grounded in our data. Retrieval quality decides answer quality. `search` and `build_prompt` are implemented in the next sections (dataset + minsearch); the skeleton below is the whole architecture.

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# search() and build_prompt() come in the next section

## 04 The Course FAQ Dataset
FAQ data from DataTalks.Club as JSON. This is our RAG knowledge base.

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
len(courses_raw), courses_raw[0]

(6,
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482})

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [13]:
from collections import Counter
Counter(d["course"] for d in documents)

Counter({'machine-learning-zoomcamp': 482,
         'data-engineering-zoomcamp': 398,
         'mlops-zoomcamp': 249,
         'llm-zoomcamp': 161,
         'stock-markets-analytics-zoomcamp': 93,
         'ai-dev-tools-zoomcamp': 73})